In [7]:
# === INPUT CONFIGURATION — edit these to run for a different line ===
LINE_NUM          = "6"                        # line identifier in shapefile
LINE_CODE         = f"bjl{LINE_NUM}"           # prefix used in output filenames
SERVICE_ID        = "weekday"                  # service_id value in trips table
SAMPLE_HOUR_START = 6                          # ] time window used to pick
SAMPLE_HOUR_END   = 10                         # ] a sample trip for speed calc
DIRECTIONS        = ["upward", "downward"]     # timetable sheet names

# Folder names
RAW_DATA_DIR      = "raw_data"                 # input data folder (relative to PROJECT_ROOT)
SUBWAY_MAP_DIR    = "beijing_subway_map"        # shapefile subfolder inside RAW_DATA_DIR
OUTPUT_DIR        = "transit_sim_inputs"        # output folder (relative to PROJECT_ROOT)

# Derived file names (no need to edit these)
STATION_CODES_FILE = "station_codes.csv"              
TIMETABLE_FILE     = "timetable.xlsx"                 
STOPS_OUT          = f"gtfs_{LINE_CODE}_stops.csv"
STOP_TIMES_OUT     = f"gtfs_{LINE_CODE}_stop_times.csv"
TRIPS_OUT          = f"gtfs_{LINE_CODE}_trips.csv"

In [8]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

try:
    from IPython.display import display
except ImportError:
    def display(*args, **kwargs): pass


### Process QGIS points to obtain stops name and locations

In [9]:
### stops file
### "stop_id","stop_name","stop_lat","stop_lon"

PROJECT_ROOT = os.path.abspath("..")

station_codes_df = pd.read_csv(os.path.join(PROJECT_ROOT, RAW_DATA_DIR, STATION_CODES_FILE))
code_to_station = dict(zip(station_codes_df["afc_code"], station_codes_df["station_name"]))

stations_map = gpd.read_file(os.path.join(PROJECT_ROOT, RAW_DATA_DIR, SUBWAY_MAP_DIR, f"{SUBWAY_MAP_DIR}.shp"))

stops_df = stations_map[
    stations_map["name"].isin(code_to_station.values()) &
    stations_map["line"].isin([LINE_NUM])
].copy()
stops_df["stop_id"]   = stops_df["name"]
stops_df["stop_name"] = stops_df["name"]
stops_df["stop_lon"]  = stops_df.geometry.x
stops_df["stop_lat"]  = stops_df.geometry.y
stops_df = stops_df[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

os.makedirs(os.path.join(PROJECT_ROOT, OUTPUT_DIR), exist_ok=True)
stops_df.to_csv(os.path.join(PROJECT_ROOT, OUTPUT_DIR, STOPS_OUT), index=False)

print(stops_df.shape)
stops_df.head(2)

(28, 4)


,stop_id,stop_name,stop_lat,stop_lon
255,haidian_wuluju,haidian_wuluju,-1600.323934,1131.556605
256,cishousi,cishousi,-1600.007659,1214.318456


### Process excel timetable to obtain stop times

In [10]:
### stop times file

timetable_file = os.path.join(PROJECT_ROOT, RAW_DATA_DIR, TIMETABLE_FILE)

schedule_excel_df_list = []
for direction in DIRECTIONS:
    schedule_excel_df = pd.read_excel(
        timetable_file,
        sheet_name=direction, index_col=[0, 1]).reset_index()

    schedule_excel_df = schedule_excel_df.melt(id_vars=[direction, "level_1"])
    schedule_excel_df = schedule_excel_df.pivot(
        index=[direction, "variable"], columns="level_1", values="value").reset_index()
    schedule_excel_df = schedule_excel_df.dropna(subset=[direction, "dept", "arr"])
    schedule_excel_df.columns = ["stop_id", "trip_id", "arrival_time", "departure_time"]
    schedule_excel_df = schedule_excel_df.sort_values(
        by=["trip_id", "arrival_time"]).reset_index(drop=True)
    schedule_excel_df["route_id"] = direction
    print(direction, schedule_excel_df.shape)
    schedule_excel_df_list.append(schedule_excel_df)

schedule_excel_df = pd.concat(schedule_excel_df_list, ignore_index=True)
print(schedule_excel_df.shape)

out_path = os.path.join(PROJECT_ROOT, OUTPUT_DIR, STOP_TIMES_OUT)
schedule_excel_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
display(schedule_excel_df.head())

# Re-filter stops to only stations that appear in the timetable.
# This removes western-extension Line 6 stations that are in station_codes.csv
# and the shapefile but have no schedule entries in our simulated segment.
timetable_stations = set(schedule_excel_df['stop_id'].unique())
stops_df = stops_df[stops_df['stop_name'].isin(timetable_stations)].copy().reset_index(drop=True)
stops_df.to_csv(os.path.join(PROJECT_ROOT, OUTPUT_DIR, STOPS_OUT), index=False)
print(f"Re-saved stops: {len(stops_df)} timetable stations → {os.path.join(PROJECT_ROOT, OUTPUT_DIR, STOPS_OUT)}")
display(stops_df[['stop_id', 'stop_name']])

upward (2214, 5)
downward (1857, 5)
(4071, 5)
Saved → /Users/suryabuchunde/Data/AAAM/transit_sim_inputs/gtfs_bjl6_stop_times.csv


,stop_id,trip_id,arrival_time,departure_time,route_id
0,jintai_lu,012011,05:56:18,05:57:08,upward
1,shilipu,012011,05:59:20,05:59:55,upward
2,qingnian_lu,012011,06:01:37,06:02:17,upward
3,dalianpo,012011,06:05:51,06:06:26,upward
4,huangqu,012011,06:08:06,06:08:41,upward


Re-saved stops: 14 timetable stations → /Users/suryabuchunde/Data/AAAM/transit_sim_inputs/gtfs_bjl6_stops.csv


,stop_id,stop_name
0,jintai_lu,jintai_lu
1,shilipu,shilipu
2,qingnian_lu,qingnian_lu
3,dalianpo,dalianpo
4,huangqu,huangqu
5,changying,changying
6,caofang,caofang
7,wuzi_xueyuan_lu,wuzi_xueyuan_lu
8,tongzhou_beiguan,tongzhou_beiguan
9,beiyunhexi,beiyunhexi


In [11]:
### calculate headway and train speed
### uses first available downward trip — no hardcoded trip_id

tmp = schedule_excel_df.copy()
tmp["arrival_time"]   = pd.to_datetime(tmp["arrival_time"])
tmp["departure_time"] = pd.to_datetime(tmp["departure_time"])

# pick first downward trip in the configured time window
downward_trips = tmp[
    (tmp["route_id"] == DIRECTIONS[1]) &
    (tmp["arrival_time"].dt.hour >= SAMPLE_HOUR_START) &
    (tmp["arrival_time"].dt.hour < SAMPLE_HOUR_END)
]["trip_id"].unique()

if len(downward_trips) == 0:
    print(f"No {DIRECTIONS[1]} trips found in {SAMPLE_HOUR_START}-{SAMPLE_HOUR_END}h window")
else:
    sample_trip = downward_trips[0]
    tmp2 = tmp[(tmp["trip_id"] == sample_trip) & (tmp["route_id"] == DIRECTIONS[1])].copy()
    tmp3 = tmp2.merge(stops_df[["stop_id", "stop_lon", "stop_lat"]], how="left", on="stop_id")
    tmp3["dist"]        = np.sqrt((tmp3["stop_lon"] - tmp3["stop_lon"].shift(-1))**2 +
                                   (tmp3["stop_lat"] - tmp3["stop_lat"].shift(-1))**2)
    tmp3["travel_time"] = (tmp3["arrival_time"].shift(-1) - tmp3["departure_time"]).dt.seconds
    tmp3["speed"]       = tmp3["dist"] / tmp3["travel_time"]
    print(f"Sample trip: {sample_trip}")
    display(tmp3)

Sample trip: 011049


/var/folders/s6/xq9sh3g514jdt4d6sp2mndmw0000gn/T/ipykernel_62271/262012686.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tmp["arrival_time"]   = pd.to_datetime(tmp["arrival_time"])
/var/folders/s6/xq9sh3g514jdt4d6sp2mndmw0000gn/T/ipykernel_62271/262012686.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tmp["departure_time"] = pd.to_datetime(tmp["departure_time"])


,stop_id,trip_id,arrival_time,departure_time,route_id,stop_lon,stop_lat,dist,travel_time,speed
0,lucheng,011049,2026-06-18 06:41:38,2026-06-18 06:42:23,downward,3809.324365,-1789.369551,59.946317,97.0,0.618003
1,dongxiayuan,011049,2026-06-18 06:44:00,2026-06-18 06:44:35,downward,3749.386372,-1790.368517,60.945147,105.0,0.580430
2,haojiafu,011049,2026-06-18 06:46:20,2026-06-18 06:46:55,downward,3688.449413,-1789.369551,59.946317,84.0,0.713647
3,beiyunhedong,011049,2026-06-18 06:48:19,2026-06-18 06:48:19,downward,3628.511421,-1790.368517,59.946317,112.0,0.535235
4,beiyunhexi,011049,2026-06-18 06:50:11,2026-06-18 06:50:46,downward,3568.573429,-1789.369551,178.096432,231.0,0.770980
5,tongzhou_beiguan,011049,2026-06-18 06:54:37,2026-06-18 06:55:12,downward,3438.707778,-1667.495633,85.415102,167.0,0.511468
6,wuzi_xueyuan_lu,011049,2026-06-18 06:57:59,2026-06-18 06:58:39,downward,3354.797853,-1651.531166,65.716423,136.0,0.483209
7,caofang,011049,2026-06-18 07:00:55,2026-06-18 07:01:40,downward,3289.081430,-1651.531166,76.198907,111.0,0.686477
8,changying,011049,2026-06-18 07:03:31,2026-06-18 07:04:06,downward,3212.888475,-1652.483578,60.954364,125.0,0.487635
9,huangqu,011049,2026-06-18 07:06:11,2026-06-18 07:06:46,downward,3151.934111,-1652.483578,59.057220,100.0,0.590572


### make trips table

In [12]:
trips_df = schedule_excel_df[["route_id", "trip_id"]].drop_duplicates().copy()
trips_df["service_id"] = SERVICE_ID

out_path = os.path.join(PROJECT_ROOT, OUTPUT_DIR, TRIPS_OUT)
trips_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
trips_df.head()

Saved → /Users/suryabuchunde/Data/AAAM/transit_sim_inputs/gtfs_bjl6_trips.csv


,route_id,trip_id,service_id
0,upward,012011,weekday
14,upward,012056,weekday
28,upward,012113,weekday
42,upward,012140,weekday
56,upward,022012,weekday
